# Protocol đánh giá anomaly (chung)

Chuẩn hóa metric để so mọi phương pháp (ETS / Chronos / IF) trên **cùng chuỗi**.

**Detection:** synthetic injection recall, alerts/năm, precision@k

**Forecast (khi có yhat):** MASE (mùa 12), interval coverage, pinball loss

**Win rule vs ETS**
- Mean recall trên `{x2.0, x0.5}` cải thiện **≥ 0.05**, **VÀ**
- `WARN+CRITICAL` alerts/năm **≤ 2×** baseline


## 1. Cấu hình & thư viện

Đọc summary ETS từ `reports/alert_backtest/*_summary.json` (sinh bởi `rice_alert_prototype.ipynb` batch).


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
if not (ROOT / "reports" / "alert_backtest").exists():
    ROOT = Path(".").resolve()

ALERT_DIR = ROOT / "reports" / "alert_backtest"
EVAL_DIR = ROOT / "reports" / "eval"
INJECT_FACTORS = [1.2, 1.5, 2.0, 3.0, 0.7, 0.5, 0.3]
KEY_FACTORS = ["x2.0", "x0.5"]
WARN = 3.0

print("ALERT_DIR =", ALERT_DIR, "exists:", ALERT_DIR.exists())
print("EVAL_DIR =", EVAL_DIR)


ALERT_DIR = C:\Homework\Code File\ML docs\Anomaly Detection\reports\alert_backtest exists: True
EVAL_DIR = C:\Homework\Code File\ML docs\Anomaly Detection\reports\eval


## 2. Hàm metric & win rule

Định nghĩa chuẩn dùng chung cho các notebook so sánh.


In [2]:
def mase(y: pd.Series, yhat: pd.Series, season: int = 12) -> float:
    y = y.loc[yhat.index].astype(float)
    yhat = yhat.astype(float)
    mask = y.notna() & yhat.notna()
    y, yhat = y[mask], yhat[mask]
    if len(y) < season + 1:
        return float("nan")
    naive = (y - y.shift(season)).abs().iloc[season:].mean()
    if naive == 0 or np.isnan(naive):
        return float("nan")
    return float((y - yhat).abs().mean() / naive)


def interval_coverage(y: pd.Series, lo: pd.Series, hi: pd.Series) -> float:
    idx = y.index.intersection(lo.index).intersection(hi.index)
    y, lo, hi = y.loc[idx], lo.loc[idx], hi.loc[idx]
    ok = y.notna() & lo.notna() & hi.notna()
    if ok.sum() == 0:
        return float("nan")
    return float(((y[ok] >= lo[ok]) & (y[ok] <= hi[ok])).mean())


def pinball_loss(y: pd.Series, qhat: pd.Series, q: float) -> float:
    idx = y.index.intersection(qhat.index)
    y, qhat = y.loc[idx].astype(float), qhat.loc[idx].astype(float)
    ok = y.notna() & qhat.notna()
    e = y[ok] - qhat[ok]
    return float(np.mean(np.where(e >= 0, q * e, (q - 1) * e)))


def precision_at_k(scores: pd.Series, labels: pd.Series, k: int = 10) -> float:
    # labels: 1 = anomaly. Rank by |score| descending.
    df = pd.DataFrame({"score": scores.abs(), "y": labels}).dropna()
    if df.empty:
        return float("nan")
    top = df.nlargest(min(k, len(df)), "score")
    return float(top["y"].mean())


def mean_key_recall(recall: dict) -> float:
    vals = []
    for col_rec in recall.values():
        for f in KEY_FACTORS:
            if f in col_rec:
                vals.append(col_rec[f])
    return float(np.mean(vals)) if vals else float("nan")


def beats_baseline(candidate: dict, baseline: dict) -> dict:
    c_rec = mean_key_recall(candidate.get("synthetic_recall", {}))
    b_rec = mean_key_recall(baseline.get("synthetic_recall", {}))
    c_rate = candidate.get("alerts_per_year", {}).get("WARN_or_CRITICAL", np.nan)
    b_rate = baseline.get("alerts_per_year", {}).get("WARN_or_CRITICAL", np.nan)
    better_recall = (c_rec - b_rec) >= 0.05 if np.isfinite(c_rec) and np.isfinite(b_rec) else False
    if np.isfinite(c_rate) and np.isfinite(b_rate) and b_rate > 0:
        rate_ok = c_rate <= 2.0 * b_rate
    elif np.isfinite(c_rate) and np.isfinite(b_rate):
        rate_ok = c_rate <= max(2.0, b_rate)
    else:
        rate_ok = False
    return {
        "candidate_key_recall": round(c_rec, 3) if np.isfinite(c_rec) else None,
        "baseline_key_recall": round(b_rec, 3) if np.isfinite(b_rec) else None,
        "candidate_alerts_per_year": c_rate,
        "baseline_alerts_per_year": b_rate,
        "better_recall": bool(better_recall),
        "rate_ok": bool(rate_ok),
        "wins": bool(better_recall and rate_ok),
        "rule": "key_recall(x2,x0.5) +0.05 AND alerts/year <= 2x baseline",
    }


## 3. Đọc summary ETS & ghi baseline JSON

Đọc `reports/alert_backtest/*_summary.json`, tính key recall, ghi
`reports/eval/baseline_ets.json` (machine-readable). Bảng kết quả xem ở cell dưới —
**không** generate file `.md` bằng code.


In [3]:
def load_ets_summaries() -> list[dict]:
    return [json.loads(p.read_text(encoding="utf-8")) for p in sorted(ALERT_DIR.glob("*_summary.json"))]


EVAL_DIR.mkdir(parents=True, exist_ok=True)
summaries = load_ets_summaries()
if not summaries:
    raise SystemExit(
        f"No ETS summaries in {ALERT_DIR}. "
        "Chạy batch trong rice_alert_prototype.ipynb trước (RUN_BATCH=True)."
    )

payload = {
    "win_rule": "key_recall(x2,x0.5)+0.05 AND alerts/year<=2x baseline",
    "series": {
        s["series"]: {
            "alerts_per_year": s["alerts_per_year"],
            "key_recall": mean_key_recall(s.get("synthetic_recall", {})),
            "synthetic_recall": s.get("synthetic_recall", {}),
        }
        for s in summaries
    },
}
(EVAL_DIR / "baseline_ets.json").write_text(
    json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Wrote {EVAL_DIR / 'baseline_ets.json'}")
print("Win rule:", payload["win_rule"])


Wrote C:\Homework\Code File\ML docs\Anomaly Detection\reports\eval\baseline_ets.json
Win rule: key_recall(x2,x0.5)+0.05 AND alerts/year<=2x baseline


## 4. Bảng baseline (hiển thị trong notebook)

Key recall = trung bình recall tại x2.0 và x0.5 trên lượng & trị giá.


In [4]:
rows = []
for name, s in payload["series"].items():
    rows.append(
        {
            "series": name,
            "WARN+CRIT/năm": s["alerts_per_year"]["WARN_or_CRITICAL"],
            "key_recall": s["key_recall"],
        }
    )
display(pd.DataFrame(rows).set_index("series").round(3))
print("Win rule:", payload["win_rule"])


,WARN+CRIT/năm,key_recall
series,,
dầu thô,1.00,0.360
gạo,0.15,0.457
sắt thép các loại,0.46,0.764
xăng dầu các loại,1.08,0.762


Win rule: key_recall(x2,x0.5)+0.05 AND alerts/year<=2x baseline
